# 07 — Baseline Comparison (Exp 2)

Trains all four of this replication's pretrain models -- `AttnLOB`, `FCLOB`, `ConvLOB`, `DeepLOB` (paper Table I) -- on the exact same data/split with the exact same *untuned* default `TrainConfig`, then evaluates each once on the held-out test split. The point is a fair, apples-to-apples comparison: none of the four get individually tuned here, so differences reflect architecture, not hyperparameter search luck.

This intentionally compares against notebook 05's *untuned* Attn-LOB result (Exp 1), not notebook 06's tuned winner (Exp 1b) -- tuning only Attn-LOB and then comparing it to untuned baselines would stack the deck. Exp 1b's number is still worth knowing as a ceiling reference, just not the one used for this comparison.

Each model's default config already matches this replication's data (`n_levels=10`, and for FC-LOB specifically `window_T=50`, which is also `FeatureConfig`'s default window length) -- see [Attn-LOB Model](../docs/attn_lob_model.md) and the model docstrings for the architecture details and paper-fidelity notes for each.

In [ ]:
from __future__ import annotations

import pathlib
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split
from paper_replication.models import (
    FCLOB,
    AttnLOB,
    AttnLOBConfig,
    ConvLOB,
    ConvLOBConfig,
    DeepLOB,
    DeepLOBConfig,
    FCLOBConfig,
)
from paper_replication.training import TrainConfig, evaluate, train_model
from paper_replication.training.dataset import AttnLOBTorchDataset

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"
CHECKPOINT_DIR = pathlib.Path("../data/processed")

torch.manual_seed(0)

## Build the dataset and split it chronologically

Same pipeline and split as notebooks 02/04/05/06.

In [ ]:
dataset = build_feature_dataset(LOB_PATH, FeatureConfig())
splits = chronological_split(dataset)

for name, split in splits.items():
    print(f"{name:>5}: n={len(split)}")

## Train all four models

Each gets its own `TrainConfig` (only `checkpoint_path` differs between them) with the same untuned defaults as Exp 1: `batch_size=256`, `learning_rate=1e-3`, `weight_decay=0.0`, `patience=5`, `seed=0`, device auto-selected. Each model's own default `*Config` is used unchanged.

In [ ]:
MODEL_SPECS = [
    ("AttnLOB", AttnLOB, AttnLOBConfig()),
    ("FCLOB", FCLOB, FCLOBConfig()),
    ("ConvLOB", ConvLOB, ConvLOBConfig()),
    ("DeepLOB", DeepLOB, DeepLOBConfig()),
]

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
results = {}

for name, model_cls, model_config in MODEL_SPECS:
    model = model_cls(model_config)
    n_params = sum(p.numel() for p in model.parameters())
    checkpoint_path = CHECKPOINT_DIR / f"{name.lower()}_baseline_checkpoint.pt"
    train_config = TrainConfig(
        batch_size=256,
        learning_rate=1e-3,
        seed=0,
        checkpoint_path=str(checkpoint_path),
    )

    start = time.time()
    history = train_model(
        model,
        splits["train"],
        splits["val"],
        train_config,
        num_classes=model_config.num_classes,
    )
    elapsed = time.time() - start
    best_epoch = min(history, key=lambda r: r["val_loss"])

    results[name] = {
        "model": model,
        "train_config": train_config,
        "history": history,
        "best_epoch": best_epoch,
        "n_params": n_params,
        "elapsed": elapsed,
    }
    print(
        f"{name:>8}: {n_params:>9,} params  "
        f"ran {len(history):2d} epochs in {elapsed:5.1f}s  "
        f"best_val_loss={best_epoch['val_loss']:.4f}  best_val_f1={best_epoch['f1']:.4f}"
    )

## Evaluate each model on the held-out test split -- exactly once per model

`splits["test"]` was never touched during any of the four training runs above.

In [ ]:
for name, r in results.items():
    device = torch.device(r["train_config"].device)
    loader = DataLoader(
        AttnLOBTorchDataset(splits["test"]), batch_size=256, shuffle=False
    )
    metrics = evaluate(r["model"], loader, device, num_classes=3)
    r["test_metrics"] = metrics
    print(
        f"{name:>8}: precision={metrics['precision']:.4f}  recall={metrics['recall']:.4f}  "
        f"f1={metrics['f1']:.4f}  accuracy={metrics['accuracy']:.4f}"
    )

## Compare against the paper's Table I

Paper's numbers are their Ping An Bank Co. results (Table I). Not a like-for-like comparison in absolute terms (see notebook's intro), but the *relative ordering* of the four models is a real, checkable claim: paper's own text calls Attn-LOB's learned representation the best performer among these four (H1's premise).

In [ ]:
PAPER_TABLE_I = {
    "FCLOB": dict(precision=0.6315, recall=0.5419, f1=0.5660),
    "ConvLOB": dict(precision=0.5851, recall=0.5230, f1=0.4984),
    "DeepLOB": dict(precision=0.7856, recall=0.6699, f1=0.7118),
    "AttnLOB": dict(precision=0.7663, recall=0.7019, f1=0.7284),
}

print(
    f"{'Model':>8} {'Ours P':>8} {'Paper P':>8} {'Ours R':>8} {'Paper R':>8} {'Ours F1':>8} {'Paper F1':>9} {'Delta F1':>9}"
)
for name in ["FCLOB", "ConvLOB", "DeepLOB", "AttnLOB"]:
    ours = results[name]["test_metrics"]
    paper = PAPER_TABLE_I[name]
    delta = ours["f1"] - paper["f1"]
    print(
        f"{name:>8} {ours['precision']:>8.4f} {paper['precision']:>8.4f} "
        f"{ours['recall']:>8.4f} {paper['recall']:>8.4f} "
        f"{ours['f1']:>8.4f} {paper['f1']:>9.4f} {delta:>+9.4f}"
    )

our_order = sorted(results, key=lambda n: -results[n]["test_metrics"]["f1"])
paper_order = sorted(PAPER_TABLE_I, key=lambda n: -PAPER_TABLE_I[n]["f1"])
print(f"\nour F1 ranking:   {our_order}")
print(f"paper F1 ranking: {paper_order}")
print(f"same ordering: {our_order == paper_order}")

## F1 comparison chart

In [ ]:
names = ["FCLOB", "ConvLOB", "DeepLOB", "AttnLOB"]
ours_f1 = [results[n]["test_metrics"]["f1"] for n in names]
paper_f1 = [PAPER_TABLE_I[n]["f1"] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - width / 2, ours_f1, width, label="ours (untuned, BTC/USDT)")
ax.bar(x + width / 2, paper_f1, width, label="paper (Table I, Ping An Bank)")
ax.axhline(1 / 3, color="gray", linestyle="--", linewidth=1, label="chance (1/3)")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel("F1")
ax.set_title("Pretrain classification F1: ours vs. paper")
ax.legend()
fig.tight_layout()
plt.show()

## Summary

**The ordering is completely inverted from the paper.** Paper: AttnLOB > DeepLOB > FCLOB > ConvLOB (by F1). Ours: ConvLOB > DeepLOB > FCLOB > AttnLOB. Untuned, on this data, the paper's best-reported architecture is *our worst performer*, and its worst-reported baseline (Conv-LOB, 20K params -- by far the smallest of the four) is *our best*.

This isn't noise -- the gap is real and the pattern makes sense once you look at what "untuned" means per model. `AttnLOB`'s `dropout` defaults to `0.0` (paper Fig. 1 doesn't specify one), and Exp 1b already showed dropout=0.3 alone moves its F1 from 0.4948 to 0.5224 on this exact data. None of the other three baselines got any equivalent regularization pass here either, but attention is the highest-capacity, most overfitting-prone piece of the four architectures, so it's the one that suffers most from having none. ConvLOB's causal dilated-residual stack, at 20K params, simply has far less room to overfit 35K training samples in the first place -- that's plausibly *why* it wins here, not because it's a "better" architecture in any general sense.

Two things this experiment does *not* claim:
- **Not evidence against the paper's finding** -- the paper's own numbers came from (presumably) tuned models; this is an untuned-vs-untuned comparison, and the one architecture we've tuned so far (AttnLOB, Exp 1b) already closed a meaningful chunk of its gap to the others with nothing but dropout.
- **Not a stable ranking** -- best val_loss ranking (DeepLOB < ConvLOB < FCLOB < AttnLOB) and best test F1 ranking (ConvLOB < DeepLOB < FCLOB < AttnLOB, reading ascending) already disagree slightly on DeepLOB vs. ConvLOB -- expected noise from a single train/val/test split, not a sign anything is broken.

Directly relevant to [H1](../docs/hypotheses.md#h1-automatic-lob-representation-learning-improves-market-making): the replication framework's success criterion "reproduced metrics follow the same relative ordering reported by the authors" does **not** hold here, at least not in this untuned form. The natural next step, if it's worth the compute, is repeating this comparison with an equivalent tuning pass (dropout/weight_decay/LR search, matching what Exp 1b did for AttnLOB) applied to all four models before ranking them -- that would test whether the paper's ordering re-emerges once every model gets a fair shot, or whether it's something more structural about this dataset that favors simpler architectures regardless of tuning.